In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("spam_train_cleaned.csv")
test_df = pd.read_csv("spam_test_cleaned.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

display(train_df.head())

Train shape: (31311, 16)
Test shape : (7828, 16)


,label,urls,hour,combined_text,capital_letter_count,capital_ratio,exclamation_count,question_count,special_char_count,day_of_week_Friday,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
0,0,1,2,volunteers are needed for alumni phonothon 200...,37,0.055306,0,0,2,0,0,0,0,0,0,1
1,0,0,0,re opensuse opensuse and faxes on sunday 10 fe...,85,0.048935,0,2,42,0,0,0,0,0,0,1
2,0,1,2,re r matching a period in grep on 08 05 2008 0...,69,0.045128,0,4,114,0,0,0,0,0,0,1
3,1,1,17,fast and safe male enhancement huge love gun i...,11,0.034700,3,0,0,0,0,0,0,1,0,0
4,0,1,3,re python dev documentation reorganization was...,44,0.026113,0,0,94,0,0,0,0,0,0,1


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix


# X and y


X = train_df.drop(columns=["label"])
y = train_df["label"]

X_test_raw = test_df.drop(columns=["label"])
y_test = test_df["label"]



# Train / Validation split


X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)



# Text + other features


TEXT_COL = "combined_text"

NUMERIC_COLS = [
    col for col in X.columns
    if col != TEXT_COL
]



# TF-IDF for text


tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_text_train = tfidf.fit_transform(
    X_train_raw[TEXT_COL]
)

X_text_val = tfidf.transform(
    X_val_raw[TEXT_COL]
)

X_text_test = tfidf.transform(
    X_test_raw[TEXT_COL]
)



# Combine text + features


X_train = hstack([
    X_text_train,
    csr_matrix(X_train_raw[NUMERIC_COLS].values)
])

X_val = hstack([
    X_text_val,
    csr_matrix(X_val_raw[NUMERIC_COLS].values)
])

X_test = hstack([
    X_text_test,
    csr_matrix(X_test_raw[NUMERIC_COLS].values)
])


print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (25048, 314)
Validation: (6263, 314)
Test: (7828, 314)


In [ ]:
# Defualt decision tree
from sklearn.tree import DecisionTreeClassifier

dt_default = DecisionTreeClassifier(
    random_state=42
)

dt_default.fit(X_train, y_train) # the training uses the full 39k not only 8k samples as in lazy classifer

print("Default Decision Tree trained!")

Default Decision Tree trained!


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_train_pred_dt = dt_default.predict(X_train)
y_val_pred_dt = dt_default.predict(X_val)


print("========== DEFAULT DECISION TREE ==========")

print("\nTRAINING PERFORMANCE")
print("Accuracy :", round(accuracy_score(y_train, y_train_pred_dt), 4))
print("Precision:", round(precision_score(y_train, y_train_pred_dt), 4))
print("Recall   :", round(recall_score(y_train, y_train_pred_dt), 4))
print("F1 Score :", round(f1_score(y_train, y_train_pred_dt), 4))


print("\nVALIDATION PERFORMANCE")
print("Accuracy :", round(accuracy_score(y_val, y_val_pred_dt), 4))
print("Precision:", round(precision_score(y_val, y_val_pred_dt), 4))
print("Recall   :", round(recall_score(y_val, y_val_pred_dt), 4))
print("F1 Score :", round(f1_score(y_val, y_val_pred_dt), 4))


print("\nCONFUSION MATRIX")
print(confusion_matrix(y_val, y_val_pred_dt))


print("\nCLASSIFICATION REPORT")
print(classification_report(y_val, y_val_pred_dt))

========== DEFAULT DECISION TREE ==========

TRAINING PERFORMANCE
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

VALIDATION PERFORMANCE
Accuracy : 0.9805
Precision: 0.9801
Recall   : 0.9851
F1 Score : 0.9826

CONFUSION MATRIX
[[2700   70]
 [  52 3441]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.98      0.97      0.98      2770
           1       0.98      0.99      0.98      3493

    accuracy                           0.98      6263
   macro avg       0.98      0.98      0.98      6263
weighted avg       0.98      0.98      0.98      6263



The tree is essentially memorizing the training data, while performance is slightly lower on unseen validation data.
- the default setup 

<code>DecisionTreeClassifier(
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1
)</code>

In [5]:
# Experminet with diffrent tree depth


from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

depth_values = [5, 10, 15, 20, 30, None]

depth_results = []

for depth in depth_values:

    dt = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    # Train on the full training split
    dt.fit(X_train, y_train)

    # Predictions
    train_pred = dt.predict(X_train)
    val_pred = dt.predict(X_val)

    # Metrics
    train_accuracy = accuracy_score(y_train, train_pred)
    val_accuracy = accuracy_score(y_val, val_pred)

    train_f1 = f1_score(y_train, train_pred)
    val_f1 = f1_score(y_val, val_pred)

    # Store results
    depth_results.append({
        "Max Depth": depth,
        "Train Accuracy": train_accuracy,
        "Validation Accuracy": val_accuracy,
        "Train F1": train_f1,
        "Validation F1": val_f1,
        "F1 Gap": train_f1 - val_f1
    })


# Results table
depth_results_df = pd.DataFrame(depth_results)

display(depth_results_df.round(4))

,Max Depth,Train Accuracy,Validation Accuracy,Train F1,Validation F1,F1 Gap
0,5.0,0.9705,0.9692,0.9737,0.9725,0.0012
1,10.0,0.9909,0.9796,0.9919,0.9817,0.0101
2,15.0,0.9947,0.9797,0.9952,0.9819,0.0133
3,20.0,0.9968,0.9791,0.9971,0.9813,0.0158
4,30.0,0.9986,0.9797,0.9988,0.9819,0.0169
5,NaN,1.0000,0.9805,1.0000,0.9826,0.0174


The default Decision Tree showed signs of overfitting, achieving 100% training F1 compared with 98.26% validation F1. The max_depth experiment confirmed this behavior. As tree depth increased, training F1 continuously increased, but validation F1 improved only slightly and then remained relatively stable. The train-validation gap also increased from 0.0012 at depth 5 to 0.0174 for the unrestricted tree. This indicates that increasing tree complexity allowed the model to fit the training data more closely without providing a comparable improvement in generalization.

 Note
- Decision Trees can achieve strong classification performance on this dataset, but an unrestricted tree is prone to overfitting. Limiting tree depth improves control over model complexity, although the validation improvement from deeper trees is very small.